# Answer

Using the vector db created from `ingest.py` we answer queries

In [ ]:
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
VECTORDB_PATH = "../irc_vectordb"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
GPT_MODEL = "gpt-4.1-nano"
load_dotenv(override=True)

In [ ]:
embeddings = HuggingFaceEmbeddings(model=EMBEDDING_MODEL)
vectordb = Chroma(persist_directory=VECTORDB_PATH, embedding_function=embeddings)

In [ ]:
retriever = vectordb.as_retriever()

In [ ]:
retriever.invoke("What is section 1031?")

In [ ]:
retriever.invoke("§ 1031") # § = "section" in the IRC pdf

Proper documents arent being retrieved

We need to try a different strategy so the proper documents are retrieved

I changed the Ingest strategy.
1. I am using a different file format
2. I cleaned the data before saving to vectordb

Lets see how it is now

In [ ]:
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

VECTORDB_PATH = "../irc_xml_vectordb"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
GPT_MODEL = "gpt-4.1-nano"
load_dotenv(override=True)

In [ ]:
embeddings = HuggingFaceEmbeddings(model=EMBEDDING_MODEL)
vectordb = Chroma(persist_directory=VECTORDB_PATH, embedding_function=embeddings)

In [ ]:
retriever = vectordb.as_retriever()

In [ ]:
retriever.invoke("What is section 1031")

Well that didnt work as well as I would've hoped. Lets try something else

In [ ]:
import re
import json
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.schema import Document

VECTORDB_PATH = "../irc_xml_vectordb"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
GPT_MODEL = "gpt-4.1-nano"
load_dotenv(override=True)

In [ ]:
def load_chunks(input_path="../Internal Revenue Code/irc_chunks.json"):
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    chunks = [Document(page_content=d["text"], metadata=d["metadata"]) for d in data]
    print(f"Loaded {len(chunks)} chunks from {input_path}")
    return chunks

chunks = load_chunks()

# Vector Retriever
embeddings = HuggingFaceEmbeddings(model=EMBEDDING_MODEL)
vectordb = Chroma(persist_directory=VECTORDB_PATH, embedding_function=embeddings)

vector_retriever = vectordb.as_retriever()

# BM25 Retriever 
bm25_retriever = BM25Retriever.from_documents(chunks)

# Ensemble Retriever 

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.6, 0.4]   # weight BM25 higher for section number queries
)

In [ ]:
def test_retrieval(query):
    print(f"\nQuery: '{query}'")
    print("=" * 60)

    # BM25 alone
    bm25_results = bm25_retriever.invoke(query)
    print("\nBM25 results:")
    for doc in bm25_results:
        print(f"  § {doc.metadata['section']} — {doc.metadata['heading']}")

    # Vector alone  
    vector_results = vector_retriever.invoke(query)
    print("\nVector results:")
    for doc in vector_results:
        print(f"  § {doc.metadata['section']} — {doc.metadata['heading']}")

    # Combined
    ensemble_results = ensemble_retriever.invoke(query)
    print("\nEnsemble results:")
    for doc in ensemble_results:
        print(f"  § {doc.metadata['section']} — {doc.metadata['heading']}")

test_retrieval("what does section 1 state?")

It's not retrieving relevant chunks - Trying a new strategy

In [ ]:
def smart_retrieve(query, vectorstore, ensemble_retriever):
    """
    If the query mentions a specific section number, bypass retrieval
    entirely and pull directly from vectorstore using metadata filter.
    Otherwise fall back to ensemble retrieval.
    """
    # Detect section number in query
    # Matches: "section 1031", "§ 1031", "sec 1031", "1031" alone
    match = re.search(
        r'(?:section|sec|§)\s*(\d+[A-Z]?)|^(\d+[A-Z]?)$',
        query.strip(),
        re.IGNORECASE
    )

    if match:
        section_num = (match.group(1) or match.group(2)).upper()
        print(f"  Section detected: § {section_num} — using metadata filter")

        results = vectorstore.get(where={"section": section_num})

        if results and results["documents"]:
            print(f"  Found {len(results['documents'])} chunks for § {section_num}")
            return [
                Document(page_content=doc, metadata=meta)
                for doc, meta in zip(results["documents"], results["metadatas"])
            ]
        else:
            print(f"  § {section_num} not found in vectorstore — falling back to ensemble")

    # No section number — use ensemble
    print("  No section detected — using ensemble retrieval")
    return ensemble_retriever.invoke(query)


# Test it
queries = [
    # "what is section 1031",
    # "§ 1031",
    # "like kind exchange",
    #"what is the capital gains tax rate",
    "section 401k retirement plans",
]

for q in queries:
    print(f"\nQuery: '{q}'")
    print("=" * 50)
    results = smart_retrieve(q, vectordb, ensemble_retriever)
    for doc in results:
        # print(f"  § {doc.metadata['section']} — {doc.metadata['heading']}")
        print(f"  {doc.page_content}")
        print()

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr

GPT_MODEL = "gpt-4.1-nano"
SYSTEM_PROMPT = """"
You are a knowledgable and friendly assistant.
You are speaking with a user about the Internal Revenue Code.
If relevant, use the given context to answer any questions.
If the given context is used to answer a question, then quote the context and the section.
If you do not know the answer, then say so.
Always mention that you are not a financial expert, and the user must consult with a professional before taking action

Context:
{context}
"""

llm = ChatOpenAI(temperature=0, model_name=GPT_MODEL)

In [ ]:
def answer_question(question: str, history):
    docs = smart_retrieve(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    print(context)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [ ]:
answer_question("what is section 5", [])

In [ ]:
gr.ChatInterface(answer_question).launch()

In [ ]:
gr.close_all()